# Introduction - Heat Distribution Example


The *heat distribution* example models how temperature diffuses over time in a given domain.
It serves as a compact but representative workload for studying stencil computations, numerical methods, and GPU performance engineering.

<div style="text-align: center;">
  <img src="./img/heat_propagation.gif" width="720" style="background-color:white" alt="Heat Distribution GIF">
</div>

## Conceptual Connection Points


**If you have a background in *numerics*:**
This is a *finite volume* or *finite difference* discretization of the heat equation $\frac{\partial u}{\partial t} = \alpha \nabla^2 u$ using *central finite differences* for the spatial derivative (Laplacian) and an *explicit Euler* method for time stepping on a *Cartesian grid*.
Each update computes a new value `uNew[i, j]` from the old field `u[i, j]` and its four nearest neighbors (two per cardinal direction).

**If you have a background in *computer science*:**
This is a *two-fold loop nest* applying a *local stencil operation*.
Each grid point depends only on a *localized neighborhood*—in this case, the 4 cardinal neighbors.
The pattern is usually memory-bound and well-suited for studying data movement, cache behavior, and domain decomposition across GPUs.

**If you have a background in *image processing*:**
This is a *filter* or *convolution*-like operation with a fixed kernel accessing neighboring pixels.
The 'filter' runs repeatedly over time steps, effectively blurring (diffusing) a given image in each iteration.


## Discretization

The 2D heat diffusion equation discretized on a Cartesian grid with uniform spacing $\Delta x$ and $\Delta y$ reads:

$$
\frac{\partial u}{\partial t} = \alpha \left(
    \frac{\partial^2 u}{\partial x^2} +
    \frac{\partial^2 u}{\partial y^2}
\right)
$$

An example of discretization is given below:

<div style="text-align: center;">
  <img src="./img/discretization.png" width="512" style="background-color:white" alt="Discretization">
</div>

Discretizing spatial derivatives using central finite differences and advancing in time with an explicit Euler scheme gives:

$$
u^{(n+1)}_{i,j} = u^{(n)}_{i,j} + \alpha \, \Delta t \left(
    \frac{u^{(n)}_{i+1,j} - 2u^{(n)}_{i,j} + u^{(n)}_{i-1,j}}{(\Delta x)^2} +
    \frac{u^{(n)}_{i,j+1} - 2u^{(n)}_{i,j} + u^{(n)}_{i,j-1}}{(\Delta y)^2}
\right)
$$

where:
* $u^{(n)}_{i, j}$: temperature at grid cell $(i, j)$ at time step $n$
* $\alpha $: *diffusivity constant* (material property)
* $\Delta t$: time step size
* $\Delta x, \Delta y$: grid spacings in x and y directions

## Stencil Update

<div style="text-align: center;">
  <img src="./img/2Dstencil.png" width="512" alt="Stencil diagram">
</div>

Assuming $dx = dy = dt = 1$, the discretized equation simplifies to

$$
u^{(n+1)}_{i, j} = u^{(n)}_{i, j} + \alpha \left(
    u^{(n)}_{i-1,j} + u^{(n)}_{i+1,j} + u^{(n)}_{i,j-1} + u^{(n)}_{i,j+1}
    - 4\,u^{(n)}_{i,j}
\right)
$$

Which can be implemented using this stencil application (pseudo code)

```c++
for y in 1 .. ny-2:
    for x in 1 .. nx-2:
        uNew(x, y) = u(x, y)
            + alpha * (
                    u(x - 1, y    )
                +     u(x + 1, y    )
                +     u(x,     y - 1)
                +     u(x,     y + 1)
                - 4 * u(x,     y    )
            )
```

and this actual implementation

```c++
for (size_t i1 = 1; i1 < ny - 1; ++i1) {
    for (size_t i0 = 1; i0 < nx - 1; ++i0) {
        uNew[i0 + i1 * nx] = u[i0 + i1 * nx]
            + alpha * (
                      u[(i0 - 1) +  i1      * nx]
                +     u[(i0 + 1) +  i1      * nx]
                +     u[ i0      + (i1 + 1) * nx]
                +     u[ i0      + (i1 - 1) * nx]
                - 4 * u[ i0      +  i1      * nx]
            );
    }
}
```
